# Bulk model validation — release 0.2

This notebook consolidates the equilibrium, mobility, recombination, diffusion,
band-gap-narrowing (BGN), and screening checks used before adding surfaces and
optics. The default examples are the experimental **n-InP ($8.5\times10^{18}$
cm$^{-3}$)** and **p-InP ($5.1\times10^{18}$ cm$^{-3}$)** wafers.

The distinction between a *code check* and a *material validation* is important:
charge neutrality and limiting behavior can be tested exactly, whereas mobility,
$B$, and Auger coefficients remain literature-model choices with uncertainty.

In [ ]:
from dataclasses import replace
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root / "src"))

from surpass import BulkModel, Wafer, get_material

try:
    import scienceplots
    plt.style.use(["science", "notebook", "no-latex"])
except ImportError:
    plt.style.use("default")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## 1. Editable assumptions

`tau_srh_s` is an assumed bulk SRH lifetime, not an intrinsic material
constant. BGN is shown both off and on because the bundled Palankovski model is
generic rather than dopant-specific.

In [ ]:
material_name = "InP"
temperature_k = 300.0
thickness_um = 620.0
n_doping_cm3 = 8.5e18
p_doping_cm3 = 5.1e18
tau_srh_s = 100e-9
injection_grid = np.logspace(11, 19, 181)

material = get_material(material_name)
wafers = {
    "n-InP": Wafer(material, donor_cm3=n_doping_cm3, thickness_um=thickness_um,
                    temperature_k=temperature_k),
    "p-InP": Wafer(material, acceptor_cm3=p_doping_cm3, thickness_um=thickness_um,
                    temperature_k=temperature_k),
}

display(pd.DataFrame({
    "quantity": ["Band structure / DOS", "Mobility", "Radiative / Auger", "BGN"],
    "implementation": ["FD neutrality; Vurgaftman/Ioffe values",
                       "compact Caughey-Thomas estimate",
                       "constant phenomenological coefficients",
                       "Palankovski point-charge model; optional"],
    "validation status": ["reference check", "provisional", "sensitivity required", "scenario model"],
}))
print("References carried by the preset:")
for ref in material.references:
    print(" -", ref)
print("Cautions:")
for note in material.cautions:
    print(" -", note)

## 2. Equilibrium and measured-wafer sanity checks

In [ ]:
rows = []
for label, wafer in wafers.items():
    for bgn in (False, True):
        s = BulkModel(wafer, enable_bgn=bgn).equilibrium()
        majority_eta = s.eta_n if wafer.net_doping_cm3 > 0 else s.eta_p
        rows.append({
            "wafer": label, "BGN": bgn, "Eg (eV)": s.eg_effective_ev,
            "dEg (meV)": 1e3*s.delta_eg_bgn_ev, "majority eta": majority_eta,
            "n0 (cm^-3)": s.n0_cm3, "p0 (cm^-3)": s.p0_cm3,
            "mu_n": s.mu_n_cm2_vs, "mu_p": s.mu_p_cm2_vs,
            "rho (ohm cm)": s.resistivity_ohm_cm,
            "Debye (nm)": s.debye_length_nm,
            "FD screening (nm)": s.fermi_screening_length_nm,
        })
equilibrium_table = pd.DataFrame(rows)
display(equilibrium_table.style.format(precision=3))

# One available experimental cross-check for the n wafer.
pred = BulkModel(wafers["n-InP"]).equilibrium()
measured_mu, measured_rho = 1299.0, 6.0e-4
display(pd.DataFrame({
    "quantity": ["electron mobility (cm2/Vs)", "resistivity (ohm cm)"],
    "model": [pred.mu_n_cm2_vs, pred.resistivity_ohm_cm],
    "measured": [measured_mu, measured_rho],
    "difference (%)": [100*(pred.mu_n_cm2_vs/measured_mu-1),
                       100*(pred.resistivity_ohm_cm/measured_rho-1)],
}))

## 3. Doping sweeps: mobility, screening, and BGN

In [ ]:
dopings = np.logspace(14, 20, 121)
data = {"n": [], "p": []}
for polarity in data:
    for density in dopings:
        wafer = Wafer(material, donor_cm3=density if polarity == "n" else 0,
                      acceptor_cm3=density if polarity == "p" else 0,
                      temperature_k=temperature_k)
        off = BulkModel(wafer).equilibrium()
        on = BulkModel(wafer, enable_bgn=True).equilibrium()
        data[polarity].append((off, on))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for polarity, color in [("n", "tab:blue"), ("p", "tab:red")]:
    states = data[polarity]
    mobility = [a.mu_n_cm2_vs if polarity == "n" else a.mu_p_cm2_vs for a, _ in states]
    axes[0].loglog(dopings, mobility, color=color, label=f"{polarity}-type")
    axes[1].loglog(dopings, [a.debye_length_nm for a, _ in states], ls="--", color=color)
    axes[1].loglog(dopings, [a.fermi_screening_length_nm for a, _ in states], color=color,
                   label=f"{polarity}-type FD")
    axes[2].semilogx(dopings, [1e3*b.delta_eg_bgn_ev for _, b in states], color=color,
                     label=f"{polarity}-type")
axes[0].set(xlabel="Doping (cm$^{-3}$)", ylabel="Majority mobility (cm$^2$ V$^{-1}$ s$^{-1}$)")
axes[1].set(xlabel="Doping (cm$^{-3}$)", ylabel="Screening length (nm)")
axes[2].set(xlabel="Doping (cm$^{-3}$)", ylabel="BGN (meV)")
axes[0].legend(); axes[1].legend(); axes[2].legend()
fig.tight_layout()

## 4. Injection-dependent recombination and transport

The plotted lifetimes are **effective excess-carrier lifetimes**
$\tau_\mathrm{eff}=\Delta n/\Delta R$. A small-signal or transient experiment
instead approaches the differential lifetime
$\tau_\mathrm{diff}=(d\Delta R/d\Delta n)^{-1}$; both are shown below.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
for column, (label, wafer) in enumerate(wafers.items()):
    model = BulkModel(wafer, tau_srh_s=tau_srh_s)
    states = [model.injection(x) for x in injection_grid]
    rad = np.array([x.delta_r_rad_cm3_s for x in states])
    aug = np.array([x.delta_r_auger_cm3_s for x in states])
    srh = np.array([x.delta_r_srh_cm3_s for x in states])
    total = rad + aug + srh
    axes[0, column].loglog(injection_grid, injection_grid/rad*1e9, label="radiative")
    axes[0, column].loglog(injection_grid, injection_grid/aug*1e9, label="Auger")
    axes[0, column].loglog(injection_grid, injection_grid/total*1e9, lw=2, label="total effective")
    differential = 1/np.gradient(total, injection_grid)
    axes[0, column].loglog(injection_grid, differential*1e9, ls="--", label="total differential")
    axes[1, column].loglog(injection_grid, [x.diffusion_length_um for x in states], lw=2)
    axes[0, column].set_title(label)
    axes[0, column].set_ylabel("Lifetime (ns)")
    axes[1, column].set(xlabel="$\\Delta n$ (cm$^{-3}$)", ylabel="Ambipolar diffusion length ($\\mu$m)")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()

## 5. Sensitivity to uncertain recombination coefficients

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, (label, wafer) in zip(axes, wafers.items()):
    for scale, color in [(0.3, "tab:blue"), (1.0, "black"), (3.0, "tab:red")]:
        varied = replace(material, b_rad_cm3_s=material.b_rad_cm3_s*scale,
                         c_n_cm6_s=material.c_n_cm6_s*scale,
                         c_p_cm6_s=material.c_p_cm6_s*scale)
        varied_wafer = Wafer(varied, donor_cm3=wafer.donor_cm3,
                             acceptor_cm3=wafer.acceptor_cm3,
                             thickness_um=wafer.thickness_um, temperature_k=wafer.temperature_k)
        model = BulkModel(varied_wafer, tau_srh_s=tau_srh_s)
        ax.loglog(injection_grid, [model.injection(x).tau_total_s*1e9 for x in injection_grid],
                  color=color, label=f"B, C scale = {scale:g}")
    ax.set(title=label, xlabel="$\\Delta n$ (cm$^{-3}$)")
axes[0].set_ylabel("Total effective lifetime (ns)")
axes[0].legend(fontsize=8)
fig.tight_layout()

## Interpretation and release-0.2 decision

- FD charge neutrality, screening, generalized Einstein transport, and the
  recombination bookkeeping pass internal checks.
- The compact n-InP mobility estimate agrees reasonably with the supplied
  measured mobility/resistivity; the p-type mobility still needs a wafer-specific
  Hall cross-check.
- BGN is an optional scenario, not yet a calibrated dopant-specific InP model.
- The spread under coefficient scaling is a reminder that absolute lifetime and
  diffusion length should not be reported without uncertainty bounds.
- These outputs are suitable inputs to the optical and later drift-diffusion
  stages, but they are not yet a prediction of measured PL or TRPL.